# NotebookPRO - OPTIONAL Model Fine-tuning

⚠️ **IMPORTANT: Training is NOT required!** ⚠️

## NotebookPRO uses RAG (Retrieval-Augmented Generation):
- ✅ **Works out-of-the-box** with pre-trained models (microsoft/phi-2, Mistral, etc.)
- ✅ **No training needed** - just upload your documents and start chatting
- ✅ Documents are embedded and stored in a vector database (ChromaDB)
- ✅ Relevant content is retrieved and fed to the LLM at query time

## When to use this notebook:
- 🎯 You want **even better** domain-specific responses
- 🎯 You have a **large corpus** of specialized material (e.g., medical, legal, technical)
- 🎯 You want the model to learn your specific writing style/terminology

## How RAG Works (No Training Required):
1. **Upload** lecture slides/books → Text extracted
2. **Embed** text → Converted to vectors using sentence-transformers
3. **Store** vectors → Saved in ChromaDB
4. **Query** → You ask a question
5. **Retrieve** → Most relevant chunks found
6. **Generate** → LLM creates response using retrieved context

## This Notebook (Optional Fine-tuning):
If you still want to fine-tune for maximum performance, this notebook will:
- Fine-tune a model on your specific educational content
- Use QLoRA for efficient training on Google Colab
- Export a custom model for NotebookPRO

**Skip this if you just want to use the app - it works perfectly with pre-trained models!**

In [ ]:
# Cell 1: Install Required Packages
# This will install all necessary libraries for training

!pip install -q transformers>=4.44.0
!pip install -q datasets>=2.19.0
!pip install -q peft>=0.11.0
!pip install -q trl>=0.9.0
!pip install -q bitsandbytes>=0.43.0
!pip install -q accelerate>=0.31.0
!pip install -q PyPDF2>=3.0.1
!pip install -q python-docx>=1.1.0
!pip install -q sentence-transformers>=2.7.0

print("✅ All packages installed successfully!")

In [ ]:
# Cell 2: Import Libraries

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer
from datasets import Dataset
import PyPDF2
from docx import Document
from pathlib import Path
import json
import re

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration

Modify these settings according to your needs:
- `MODEL_NAME`: Base model to fine-tune
- `OUTPUT_DIR`: Where to save the trained model
- `LEARNING_RATE`: Training learning rate
- `NUM_EPOCHS`: Number of training epochs

In [ ]:
# Cell 3: Configuration

# Model configuration
MODEL_NAME = "microsoft/phi-2"  # Can also use: "mistralai/Mistral-7B-v0.1", "meta-llama/Llama-2-7b-hf"
OUTPUT_DIR = "./notebookpro_model"

# Training hyperparameters
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
MAX_LENGTH = 1024

# LoRA configuration
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

print("Configuration set!")
print(f"Model: {MODEL_NAME}")
print(f"Output directory: {OUTPUT_DIR}")

## Data Preparation

Upload your documents (PDFs, DOCX, TXT) to Google Colab using the file upload feature or mount Google Drive.

In [ ]:
# Cell 4: Mount Google Drive (Optional but Recommended)

from google.colab import drive
drive.mount('/content/drive')

# Set your data directory (modify this path)
DATA_DIR = "/content/drive/MyDrive/NotebookPRO_Data"  # Change this to your folder

print(f"Using data directory: {DATA_DIR}")

In [ ]:
# Cell 5: Document Processing Functions

def extract_text_from_pdf(file_path):
    """Extract text from PDF file."""
    text = ""
    try:
        with open(file_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            for page in pdf_reader.pages:
                text += page.extract_text() + "\n"
    except Exception as e:
        print(f"Error extracting PDF {file_path}: {e}")
    return text

def extract_text_from_docx(file_path):
    """Extract text from DOCX file."""
    try:
        doc = Document(file_path)
        text = "\n".join([paragraph.text for paragraph in doc.paragraphs])
    except Exception as e:
        print(f"Error extracting DOCX {file_path}: {e}")
        text = ""
    return text

def extract_text_from_txt(file_path):
    """Extract text from TXT file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            text = file.read()
    except Exception as e:
        print(f"Error extracting TXT {file_path}: {e}")
        text = ""
    return text

def clean_text(text):
    """Clean and normalize text."""
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove special characters but keep punctuation
    text = re.sub(r'[^\w\s.,!?;:()\-\'\"]+', '', text)
    return text.strip()

def process_documents(data_dir):
    """Process all documents in the directory."""
    documents = []
    data_path = Path(data_dir)
    
    # Process PDFs
    for pdf_file in data_path.glob("**/*.pdf"):
        print(f"Processing: {pdf_file.name}")
        text = extract_text_from_pdf(pdf_file)
        if text:
            documents.append({
                'filename': pdf_file.name,
                'content': clean_text(text)
            })
    
    # Process DOCX
    for docx_file in data_path.glob("**/*.docx"):
        print(f"Processing: {docx_file.name}")
        text = extract_text_from_docx(docx_file)
        if text:
            documents.append({
                'filename': docx_file.name,
                'content': clean_text(text)
            })
    
    # Process TXT
    for txt_file in data_path.glob("**/*.txt"):
        print(f"Processing: {txt_file.name}")
        text = extract_text_from_txt(txt_file)
        if text:
            documents.append({
                'filename': txt_file.name,
                'content': clean_text(text)
            })
    
    print(f"\n✅ Processed {len(documents)} documents")
    return documents

print("Document processing functions ready!")

In [ ]:
# Cell 6: Create Training Dataset

def create_training_examples(documents):
    """Create training examples from documents."""
    examples = []
    
    # Define templates for different use cases
    templates = {
        'explanation': "Explain the following concept in detail:\n\n{content}\n\nExplanation:",
        'summary': "Provide a concise summary of the following:\n\n{content}\n\nSummary:",
        'qa': "Based on the following content, answer questions:\n\n{content}\n\nQuestion:",
        'notes': "Create structured study notes from the following:\n\n{content}\n\nNotes:"
    }
    
    for doc in documents:
        content = doc['content']
        
        # Split into chunks (approx 500 words per chunk)
        words = content.split()
        chunk_size = 500
        
        for i in range(0, len(words), chunk_size):
            chunk = ' '.join(words[i:i + chunk_size])
            
            if len(chunk.split()) < 50:  # Skip very short chunks
                continue
            
            # Create examples for each use case
            for use_case, template in templates.items():
                prompt = template.format(content=chunk[:800])  # Limit content length
                examples.append({
                    'text': prompt,
                    'source': doc['filename'],
                    'use_case': use_case
                })
    
    return examples

# Process your documents
documents = process_documents(DATA_DIR)

# Create training examples
training_examples = create_training_examples(documents)
print(f"✅ Created {len(training_examples)} training examples")

# Create dataset
train_dataset = Dataset.from_list(training_examples)
print(f"Dataset size: {len(train_dataset)}")
print(f"\nSample example:")
print(train_dataset[0]['text'][:300] + "...")

## Model Loading and Configuration

In [ ]:
# Cell 7: Load Base Model with Quantization

# Configure 4-bit quantization for efficient training
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load model
print(f"Loading model: {MODEL_NAME}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Set pad token if not set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Prepare model for training
model = prepare_model_for_kbit_training(model)

print("✅ Model loaded successfully!")

In [ ]:
# Cell 8: Configure LoRA

# LoRA configuration
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Adjust based on model
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM"
)

# Add LoRA adapters
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

## Training Configuration

In [ ]:
# Cell 9: Setup Training Arguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    fp16=True,
    save_total_limit=3,
    logging_steps=10,
    save_steps=100,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
)

print("Training arguments configured!")

In [ ]:
# Cell 10: Initialize Trainer

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    args=training_args,
    max_seq_length=MAX_LENGTH,
    dataset_text_field="text",
    packing=False,
)

print("✅ Trainer initialized and ready!")

## Start Training

This will take some time depending on your dataset size and number of epochs.

In [ ]:
# Cell 11: Train the Model

print("🚀 Starting training...")
print(f"Total steps: {len(train_dataset) // (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS) * NUM_EPOCHS}")

# Train
trainer.train()

print("✅ Training completed!")

## Save the Model

In [ ]:
# Cell 12: Save the Fine-tuned Model

# Save the model
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"✅ Model saved to {OUTPUT_DIR}")
print("\nTo use this model in NotebookPRO:")
print("1. Download the entire output directory")
print("2. Place it in the 'models' folder of your NotebookPRO installation")
print("3. Update the MODEL_PATH in config.py to point to this model")

## Test the Model

In [ ]:
# Cell 13: Test Inference

# Load the trained model for testing
test_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

test_model = PeftModel.from_pretrained(test_model, OUTPUT_DIR)

# Create a text generation pipeline
pipe = pipeline(
    "text-generation",
    model=test_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.7
)

print("✅ Model loaded for testing!")

In [ ]:
# Cell 14: Generate Sample Outputs

# Test prompts
test_prompts = [
    "Explain the following concept in detail:\n\nMachine learning is a subset of artificial intelligence.\n\nExplanation:",
    "Provide a concise summary of the following:\n\nDeep learning uses neural networks with multiple layers.\n\nSummary:",
]

print("Testing model with sample prompts:\n")

for i, prompt in enumerate(test_prompts, 1):
    print(f"--- Test {i} ---")
    print(f"Prompt: {prompt[:100]}...")
    
    result = pipe(prompt, do_sample=True, top_p=0.95)
    generated_text = result[0]['generated_text']
    
    # Extract only the generated part (after the prompt)
    response = generated_text[len(prompt):].strip()
    
    print(f"Response: {response}\n")

## Export Model for NotebookPRO

Download the model directory and use it in your application.

In [ ]:
# Cell 15: Zip the Model for Download

import shutil

# Create a zip file of the model
model_zip = f"{OUTPUT_DIR}.zip"
shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)

print(f"✅ Model packaged as: {model_zip}")
print("\nDownload this file and extract it in your NotebookPRO/models/ directory")
print("\nThen update config.py:")
print(f'MODEL_PATH = "./models/{Path(OUTPUT_DIR).name}"')

from google.colab import files
files.download(model_zip)